In [3]:
import pandas as pd
import numpy as np
import re
import os
from urllib.parse import urlparse

print('✅ Libraries imported successfully')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

✅ Libraries imported successfully
   pandas  : 2.3.3
   numpy   : 2.2.6


In [7]:
# ⚙️ عدّل DATA_DIR ليطابق مكان مجلد data عندك
DATA_DIR    = 'Phishing-Detection/data/raw/'
RESULTS_DIR = 'Phishing-Detection/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

FILES = {
    'Ebbu2017_1'       : os.path.join(DATA_DIR, 'Ebbu2017_1.csv'),
    'Ebbu2017_2'       : os.path.join(DATA_DIR, 'Ebbu2017_2.csv'),
    'GramBeddings1'    : os.path.join(DATA_DIR, 'GramBeddings1.csv'),
    'GramBeddings2'    : os.path.join(DATA_DIR, 'GramBeddings2.csv'),
    'Legitimate URLs_1': os.path.join(DATA_DIR, 'Legitimate URLs_1.csv'),
    'Legitimate URLs_2': os.path.join(DATA_DIR, 'Legitimate URLs_2.csv'),    
    'MendeleyPhishData': os.path.join(DATA_DIR, 'MendeleyPhishData.csv'),
    'openphish_feed'   : os.path.join(DATA_DIR, 'openphish_feed.csv'),
    'PhishTank'        : os.path.join(DATA_DIR, 'PhishTank.csv'),
}
OUTPUT_FILE = os.path.join(RESULTS_DIR, 'phishing_dataset_final.csv')

print('📂 Data directory :', DATA_DIR)
print('💾 Output file    :', OUTPUT_FILE)
print()
for name, path in FILES.items():
    exists = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'   {exists}  {name}')

📂 Data directory : Phishing-Detection/data/raw/
💾 Output file    : Phishing-Detection/results\phishing_dataset_final.csv

   ✅  Ebbu2017_1
   ✅  Ebbu2017_2
   ✅  GramBeddings1
   ✅  GramBeddings2
   ✅  Legitimate URLs_1
   ✅  Legitimate URLs_2
   ✅  MendeleyPhishData
   ✅  openphish_feed
   ✅  PhishTank


In [9]:
datasets = []   # قائمة تجميع جميع الـ DataFrames

# ── 1. Ebbu2017_1 ────────────────────────────────────────────────
print('📥 Loading Ebbu2017_1 ...')
df = pd.read_csv(FILES['Ebbu2017_1'], on_bad_lines='skip')
df = df.rename(columns={'URLs': 'url', 'Labels': 'label'})
df = df[['url', 'label']].copy()
df['label'] = df['label'].astype(int)
df['source'] = 'Ebbu2017_1'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={df["label"].sum():,}  legit={(df["label"]==0).sum():,}')

# ── 2. Ebbu2017_2 ────────────────────────────────────────────────
print('📥 Loading Ebbu2017_2 ...')
df = pd.read_csv(FILES['Ebbu2017_2'], on_bad_lines='skip')
df = df.rename(columns={'URLs': 'url', 'Labels': 'label'})
df = df[['url', 'label']].copy()
df['label'] = df['label'].astype(int)
df['source'] = 'Ebbu2017_2'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={df["label"].sum():,}  legit={(df["label"]==0).sum():,}')

# ── 3. GramBeddings1 ─────────────────────────────────────────────
# بدون header → col0=label (1=phish, 2=legit) , col1=url
print('📥 Loading GramBeddings1 ...')
df = pd.read_csv(FILES['GramBeddings1'], header=None,
                 names=['label', 'url'], on_bad_lines='skip')
df['label'] = df['label'].map({1: 1, 2: 0})  # 1→phish , 2→legit
df = df[['url', 'label']].copy()
df['source'] = 'GramBeddings1'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={df["label"].sum():,}  legit={(df["label"]==0).sum():,}')

# ── 4. GramBeddings2 ─────────────────────────────────────────────
# عمود 0 فقط = url ، جميع الروابط تصيد
print('📥 Loading GramBeddings2 ...')
df = pd.read_csv(FILES['GramBeddings2'], header=None,
                 usecols=[0], names=['url'], on_bad_lines='skip')
df['label'] = 1
df['source'] = 'GramBeddings2'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={df["label"].sum():,}  legit=0')

# ── 5. Legitimate URLs_1 ─────────────────────────────────────────
# عمود 0=index رقمي ، عمود 1=url ، جميع الروابط سليمة
print('📥 Loading Legitimate URLs_1 ...')
df = pd.read_csv(FILES['Legitimate URLs_1'], header=None,
                 usecols=[1], names=['url'], on_bad_lines='skip')
df['label'] = 0
df['source'] = 'Legitimate URLs_1'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish=0  legit={len(df):,}')
# ── 5. Legitimate URLs_2 ─────────────────────────────────────────
# عمود 0=index رقمي ، عمود 1=url ، جميع الروابط سليمة
print('📥 Loading Legitimate URLs_2 ...')
df = pd.read_csv(FILES['Legitimate URLs_2'], header=None,
                 usecols=[1], names=['url'], on_bad_lines='skip')
df['label'] = 0
df['source'] = 'Legitimate URLs_2'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish=0  legit={len(df):,}')
# ── 6. MendeleyPhishData ─────────────────────────────────────────
print('📥 Loading MendeleyPhishData ...')
df = pd.read_csv(FILES['MendeleyPhishData'], on_bad_lines='skip')
df = df.rename(columns={'URL': 'url', 'Label': 'label'})
df = df[['url', 'label']].copy()
df['label'] = df['label'].astype(int)
df['source'] = 'MendeleyPhishData'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={df["label"].sum():,}  legit={(df["label"]==0).sum():,}')

# ── 7. openphish_feed ────────────────────────────────────────────
# بدون header → عمود 0 = url ، جميع الروابط تصيد
print('📥 Loading openphish_feed ...')
df = pd.read_csv(FILES['openphish_feed'], header=None,
                 names=['url'], on_bad_lines='skip')
df['label'] = 1
df['source'] = 'openphish_feed'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={len(df):,}  legit=0')

# ── 8. PhishTank ─────────────────────────────────────────────────
# أعمدة متعددة، نأخذ عمود url فقط ، جميع الروابط تصيد
print('📥 Loading PhishTank ...')
df = pd.read_csv(FILES['PhishTank'], usecols=['url'], on_bad_lines='skip')
df['label'] = 1
df['source'] = 'PhishTank'
datasets.append(df)
print(f'   ➤ {len(df):,} rows  |  phish={len(df):,}  legit=0')

print()
print('✅ All datasets loaded!')

📥 Loading Ebbu2017_1 ...
   ➤ 20,000 rows  |  phish=10,000  legit=10,000
📥 Loading Ebbu2017_2 ...
   ➤ 55,000 rows  |  phish=5,000  legit=50,000
📥 Loading GramBeddings1 ...
   ➤ 350,000 rows  |  phish=175,125  legit=174,875
📥 Loading GramBeddings2 ...
   ➤ 20,000 rows  |  phish=20,000  legit=0
📥 Loading Legitimate URLs_1 ...
   ➤ 1,000,000 rows  |  phish=0  legit=1,000,000
📥 Loading Legitimate URLs_2 ...
   ➤ 1,000,000 rows  |  phish=0  legit=1,000,000
📥 Loading MendeleyPhishData ...
   ➤ 149,726 rows  |  phish=54,807  legit=94,919
📥 Loading openphish_feed ...
   ➤ 300 rows  |  phish=300  legit=0
📥 Loading PhishTank ...
   ➤ 55,825 rows  |  phish=55,825  legit=0

✅ All datasets loaded!


In [10]:
print('🔗 Concatenating all datasets ...')
combined = pd.concat(datasets, ignore_index=True)

print(f'\n📊 Combined shape : {combined.shape}')
print(f'   Total rows     : {len(combined):,}')
print(f'   Phishing  (1)  : {(combined["label"]==1).sum():,}')
print(f'   Legitimate(0)  : {(combined["label"]==0).sum():,}')
print()
print('📋 Rows per source:')
print(combined['source'].value_counts().to_string())

🔗 Concatenating all datasets ...

📊 Combined shape : (2650851, 3)
   Total rows     : 2,650,851
   Phishing  (1)  : 321,057
   Legitimate(0)  : 2,329,794

📋 Rows per source:
source
Legitimate URLs_1    1000000
Legitimate URLs_2    1000000
GramBeddings1         350000
MendeleyPhishData     149726
PhishTank              55825
Ebbu2017_2             55000
Ebbu2017_1             20000
GramBeddings2          20000
openphish_feed           300


In [11]:
before = len(combined)
combined.drop_duplicates(subset=['url'], keep='first', inplace=True)
combined.reset_index(drop=True, inplace=True)
after = len(combined)

print(f'🗑️  Duplicates removed : {before - after:,}')
print(f'   Before : {before:,}')
print(f'   After  : {after:,}')
print()
print(f'   Phishing  (1) : {(combined["label"]==1).sum():,}')
print(f'   Legitimate(0) : {(combined["label"]==0).sum():,}')

🗑️  Duplicates removed : 234,538
   Before : 2,650,851
   After  : 2,416,313

   Phishing  (1) : 283,400
   Legitimate(0) : 2,132,913


In [12]:
before = len(combined)

combined.dropna(subset=['url', 'label'], inplace=True)
combined['url'] = combined['url'].astype(str).str.strip()
combined = combined[combined['url'] != '']
combined = combined[combined['url'].str.lower() != 'nan']

after = len(combined)
print(f'🧹 NaN / empty rows removed : {before - after:,}')
print(f'   Remaining rows : {after:,}')

🧹 NaN / empty rows removed : 0
   Remaining rows : 2,416,313


In [13]:
def is_valid_url(url: str) -> bool:
    try:
        url = url.strip()
        if not re.match(r'^https?://', url, re.IGNORECASE):
            url_to_parse = 'https://' + url
        else:
            url_to_parse = url
        parsed = urlparse(url_to_parse)
        if not parsed.netloc:
            return False
        domain = parsed.netloc.split(':')[0]
        if '.' not in domain:
            return False
        if len(url) < 4:
            return False
        return True
    except Exception:
        return False

before = len(combined)
print('⏳ Validating URLs (this may take a moment) ...')

valid_mask   = combined['url'].apply(is_valid_url)
invalid_urls = combined[~valid_mask]
combined     = combined[valid_mask].copy()
combined.reset_index(drop=True, inplace=True)

after = len(combined)
print(f'❌ Invalid URLs removed : {before - after:,}')
print(f'   Remaining rows      : {after:,}')
if len(invalid_urls) > 0:
    print('⚠️  Sample of removed invalid URLs:')
    print(invalid_urls['url'].head(10).to_string())

⏳ Validating URLs (this may take a moment) ...
❌ Invalid URLs removed : 68
   Remaining rows      : 2,416,245
⚠️  Sample of removed invalid URLs:
58807     "https://www.hs-fresenius.de/wp-json/wp/v2/pag...
68554     mumble://:electronicchampions@eu.mumble.champ....
75471                                 mailto:www@sartak.org
76431     mailto:mail@barbara-glasner.de?subject=Anfrage...
78093                         ttp://improbable.com/ig/2009/
79164     mailto:?subject=garagentor-vergleich.de%20-%20...
87774     feed://goballycastle.com/blog/index.php?blog=2...
100244    http://ttp://xn--9y2b19kb1eutan3r1zggxaw2wfxc....
108080                             samp://SV.IRSamp.iR:7777
111051                                 mailto:wwb@zyrlx.com


In [14]:
before = len(combined)
combined['label'] = pd.to_numeric(combined['label'], errors='coerce')
combined = combined[combined['label'].isin([0, 1])].copy()
combined['label'] = combined['label'].astype(int)
after = len(combined)

print(f'🏷️  Rows with invalid labels removed : {before - after:,}')
print(f'   Remaining rows : {after:,}')
print()
print('✅ Final label distribution:')
vc = combined['label'].value_counts().rename({1:'Phishing (1)', 0:'Legitimate (0)'})
print(vc.to_string())
total = len(combined)
print(f'\n   Phishing  % : {(combined["label"]==1).sum()/total*100:.2f}%')
print(f'   Legitimate% : {(combined["label"]==0).sum()/total*100:.2f}%')

🏷️  Rows with invalid labels removed : 0
   Remaining rows : 2,416,245

✅ Final label distribution:
label
Legitimate (0)    2132847
Phishing (1)       283398

   Phishing  % : 11.73%
   Legitimate% : 88.27%


In [15]:
RANDOM_SEED = 42

combined = combined.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'🔀 Data shuffled with random_state={RANDOM_SEED}')
print(f'   Total rows : {len(combined):,}')
print()
print('📝 Sample after shuffle (first 5 rows):')
print(combined[['url', 'label']].head())

🔀 Data shuffled with random_state=42
   Total rows : 2,416,245

📝 Sample after shuffle (first 5 rows):
                     url  label
0  inside-commercial.com      0
1          buspronet.net      0
2     les-coccinelles.fr      0
3            1057.casino      0
4   strapi.scentbird.com      0


In [16]:
final_df = combined[['url', 'label']].copy()

final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print('=' * 55)
print('✅  FINAL DATASET SAVED SUCCESSFULLY')
print('=' * 55)
print(f'📄 File         : {OUTPUT_FILE}')
print(f'📊 Total rows   : {len(final_df):,}')
print(f'🎣 Phishing (1) : {(final_df["label"]==1).sum():,}')
print(f'✔️  Legit    (0) : {(final_df["label"]==0).sum():,}')
print(f'📏 File size    : {os.path.getsize(OUTPUT_FILE)/1024/1024:.2f} MB')
print('=' * 55)

✅  FINAL DATASET SAVED SUCCESSFULLY
📄 File         : Phishing-Detection/results\phishing_dataset_final.csv
📊 Total rows   : 2,416,245
🎣 Phishing (1) : 283,398
✔️  Legit    (0) : 2,132,847
📏 File size    : 80.69 MB


In [19]:
print('╔══════════════════════════════════════════════════╗')
print('║         FINAL DATASET SUMMARY REPORT            ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Total URLs       : {len(final_df):>10,}                 ║')
print(f'║  Phishing  (1)    : {(final_df["label"]==1).sum():>10,}                 ║')
print(f'║  Legitimate(0)    : {(final_df["label"]==0).sum():>10,}                 ║')
print('║  Columns          : url, label               ║')
print('║  Encoding         : UTF-8                    ║')
print('║  Duplicates       : Removed ✅               ║')
print('║  Invalid URLs     : Removed ✅               ║')
print('║  NaN values       : Removed ✅               ║')
print('║  Shuffled         : Yes (seed=42) ✅          ║')
print('╠══════════════════════════════════════════════════╣')
print('║  Sources: Legitimate URLs_1 / Legitimate URLs_2 / Ebbu2017 / GramBeddings             ║')
print('║           Legitimate_URLs / Mendeley          ║')
print('║           openphish / PhishTank  /                   ║')
print('╚══════════════════════════════════════════════════╝')
print()
print('📋 First 5 rows:')
print(final_df.head())
print()
print('📐 URL length statistics:')
final_df['url_length'] = final_df['url'].str.len()
print(final_df['url_length'].describe())

╔══════════════════════════════════════════════════╗
║         FINAL DATASET SUMMARY REPORT            ║
╠══════════════════════════════════════════════════╣
║  Total URLs       :  2,416,245                 ║
║  Phishing  (1)    :    283,398                 ║
║  Legitimate(0)    :  2,132,847                 ║
║  Columns          : url, label               ║
║  Encoding         : UTF-8                    ║
║  Duplicates       : Removed ✅               ║
║  Invalid URLs     : Removed ✅               ║
║  NaN values       : Removed ✅               ║
║  Shuffled         : Yes (seed=42) ✅          ║
╠══════════════════════════════════════════════════╣
║  Sources: Legitimate URLs_1 / Legitimate URLs_2 / Ebbu2017 / GramBeddings             ║
║           Legitimate_URLs / Mendeley          ║
║           openphish / PhishTank  /                   ║
╚══════════════════════════════════════════════════╝

📋 First 5 rows:
                     url  label
0  inside-commercial.com      0
1          bus